# Build no_UPDRS Train / Test matrices

Creates the reduced sparse matrices used by every notebook in this folder.

The questionnaire set has changed: the 13 MDS-UPDRS Part I (non-motor
experiences of daily living) items are no longer collected, so they are dropped
from both the any-input and 50%-sparse matrices.

Everything else is untouched — same rows, same `PATNO` / `EVENT_ID` / `APPRDX`
columns — so the downstream notebooks are identical to the
`with_UPDRS` versions apart from their `BASE_DIR`.

* **Source** — `with_UPDRS/{Train,Test}/Sparse_Matrix_Min{5,50}_{Train,Test}.csv`
  (falls back to the project root if `with_UPDRS/` is absent)
* **Output** — `no_UPDRS/{Train,Test}/Sparse_Matrix_{AnyInput,Min50}_{Train,Test}.csv`

### Note on the AnyInput name

The old `Min5` name implied a 5% completeness floor that the matrix does
not actually apply — it keeps every visit regardless of how many
questions were answered. The outputs here are named
`Sparse_Matrix_AnyInput_*` instead. The `with_UPDRS` source files keep
their historical `Min5` names, since that archive and its published
results were generated under them.

Run this notebook first, then `Various_M1_MiniModel.ipynb`,
`Various_M2_MiniModel.ipynb`, `Various_M3_Collab_MiniModel.ipynb` and
`Train_3SubModels.ipynb` in any order.


## Configuration

In [ ]:
import os
import pandas as pd

BASE_DIR   = os.getcwd()   # run from the repository root
PROJ_DIR   = os.path.join(os.getcwd(), "data")   # place the source sparse matrices here

# Prefer the archived copy under with_UPDRS; fall back to the project root.
SRC_ROOT = os.path.join(PROJ_DIR, "source")
if not os.path.isdir(SRC_ROOT):
    SRC_ROOT = PROJ_DIR

OUT_TRAIN = os.path.join(BASE_DIR, "Train")
OUT_TEST  = os.path.join(BASE_DIR, "Test")
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST,  exist_ok=True)

print("Source:", SRC_ROOT)
print("Output:", OUT_TRAIN, "|", OUT_TEST)

## Columns removed

The 13 MDS-UPDRS Part I items.

`Depression_GDS` (Geriatric Depression Scale) is a separate instrument and is
**kept** — only the UPDRS `Depression` item is dropped.

In [ ]:
DROP_COLS = [
    "Constipation",
    "Fatigue",
    "Lightheadedness",
    "Pain",
    "Sleepiness_Daytime",
    "Sleep_Problems_Nocturnal",
    "Urine",
    "Anxiety",
    "Apathy",
    "Cognitive_Impairment",
    "Depression",
    "Dopamine_Dysregulation_Syndrome",
    "Hallucination",
    "Hand",            # handedness: removed completely (no clinical link to PD)
]

FILES = [
    (os.path.join(SRC_ROOT, "Train", "Sparse_Matrix_Min5_Train.csv"),
     os.path.join(OUT_TRAIN, "Sparse_Matrix_AnyInput_Train.csv")),
    (os.path.join(SRC_ROOT, "Train", "Sparse_Matrix_Min50_Train.csv"),
     os.path.join(OUT_TRAIN, "Sparse_Matrix_Min50_Train.csv")),
    (os.path.join(SRC_ROOT, "Test",  "Sparse_Matrix_Min5_Test.csv"),
     os.path.join(OUT_TEST,  "Sparse_Matrix_AnyInput_Test.csv")),
    (os.path.join(SRC_ROOT, "Test",  "Sparse_Matrix_Min50_Test.csv"),
     os.path.join(OUT_TEST,  "Sparse_Matrix_Min50_Test.csv")),
]

print(f"{len(DROP_COLS)} columns will be dropped from {len(FILES)} files")

14 columns will be dropped from 4 files


## Pre-flight — check the target files are not locked

In [ ]:
# -- Pre-flight: make sure every target is writable --------------------------
# Windows locks a CSV while it is open in Excel, which would make to_csv fail
# part-way through and leave the folder half-updated. Check all targets first.
locked = []
for _, dst in FILES:
    if os.path.exists(dst):
        try:
            with open(dst, "a"):
                pass
        except PermissionError:
            locked.append(dst)

if locked:
    raise PermissionError(
        "These files are open in another program (usually Excel) and cannot be "
        "written. Close them and re-run:\n  " + "\n  ".join(locked))

print("All targets writable.")

All targets writable.


## Build

In [ ]:
for src, dst in FILES:
    df = pd.read_csv(src)
    present = [c for c in DROP_COLS if c in df.columns]
    missing = [c for c in DROP_COLS if c not in df.columns]
    out = df.drop(columns=present)
    if "Min50" in dst:
        # Visit inclusion: >= 16 of the 30 clinical items that the model uses.
        clin = [c for c in out.columns if c not in ("PATNO", "EVENT_ID", "APPRDX", "Age", "Sex")]
        keep = out[clin].notna().sum(axis=1) >= 16
        print(f"    inclusion >= 16 of {len(clin)} items: dropped {int((~keep).sum())} visits")
        out = out[keep]
    out.to_csv(dst, index=False)

    print(f"  {os.path.basename(src)}")
    print(f"    rows {len(df)}  cols {df.shape[1]} -> {out.shape[1]} "
          f"(dropped {len(present)})")
    if missing:
        print(f"    NOTE: not found in source: {missing}")
    print(f"    saved -> {dst}")

print("\n  Done.")

## Verify

In [ ]:
check = pd.read_csv(os.path.join(OUT_TRAIN, "Sparse_Matrix_AnyInput_Train.csv"))
leftover = [c for c in DROP_COLS if c in check.columns]

print("Shape:", check.shape)
print("UPDRS columns remaining:", leftover if leftover else "none")
print("Clinical features:",
      len([c for c in check.columns
           if c not in ("PATNO", "EVENT_ID", "APPRDX", "Age", "Sex")]))
print("\nColumns:")
print(list(check.columns))